# Random passport JPG folder to PDF - updated version

This version selects a random JPG from `data/passports_images/with_photo` and inserts it into a random page of a 1-10 page PDF.

In [ ]:
pip install reportlab pypdf

In [ ]:
from pathlib import Path
import random, textwrap
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas
from pypdf import PdfReader

IMAGE_DIR = Path("data/passports_images/with_photo")
OUTPUT_PDF = Path("data/pdf/PDF_Normal_Doc_10.pdf")

jpg_files = [p for p in IMAGE_DIR.rglob("*")
             if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg"}]
if not jpg_files:
    raise FileNotFoundError(f"No JPG images found in: {IMAGE_DIR.resolve()}")

selected_image = random.choice(jpg_files)
page_count = random.randint(1, 10)
image_page = random.randint(1, page_count)

print(f"Found {len(jpg_files)} JPG files")
print(f"Selected image: {selected_image}")
print(f"PDF pages: {page_count}; image page: {image_page}")

In [ ]:
WORDS = "data model random page document example simple text number system result file report process value".split()
OUTPUT_PDF.parent.mkdir(parents=True, exist_ok=True)

def random_lines(count):
    text = " ".join(random.choice(WORDS) for _ in range(count * 10))
    return textwrap.wrap(text, width=78)[:count]

def draw_lines(pdf, lines, y):
    pdf.setFont("Helvetica", 10)
    for line in lines:
        pdf.drawString(55, y, line)
        y -= 14

pdf = canvas.Canvas(str(OUTPUT_PDF), pagesize=A4)
width, height = A4
for page in range(1, page_count + 1):
    pdf.setFont("Helvetica-Bold", 15)
    pdf.drawString(55, height - 45, "Random document")

    if page == image_page:
        draw_lines(pdf, random_lines(9), height - 70)
        image = ImageReader(str(selected_image))
        image_width, image_height = image.getSize()
        scale = min(430 / image_width, 260 / image_height)
        draw_width, draw_height = image_width * scale, image_height * scale
        pdf.drawImage(image, (width - draw_width) / 2, 350, draw_width, draw_height)
        draw_lines(pdf, random_lines(16), 330)
    else:
        draw_lines(pdf, random_lines(38), height - 70)

    pdf.setFont("Helvetica", 9)
    pdf.drawCentredString(width / 2, 25, f"Page {page} of {page_count}")
    pdf.showPage()

pdf.save()
assert len(PdfReader(str(OUTPUT_PDF)).pages) == page_count
print(f"Created: {OUTPUT_PDF.resolve()}")